# Module 18 — Analytics Engineering: Dimensional Modelling & Pipelines

## What you will discover in this notebook

Every cell below runs the module's **real** implementation from
`project_solution/`. Nothing here prints a claim it has not computed.

1. That an SCD Type 2 dimension answers *"what was true then?"* — and that
   `current()` cannot.
2. That the same four sales, loaded twice, produce **different** totals when the
   as-of join is wrong — and that the first load is always correct, which is why
   the bug reaches production.
3. That an incremental refresh is measurably cheaper, and valid only for
   additive measures.
4. That a pipeline's idempotency has two halves, and implementing only the first
   makes failures permanent.
5. That the hand-built model and real DuckDB agree — the only real evidence that
   the model is not lying to you.

**One cell near the end is deliberately broken.** Fixing it is the exercise.

## Setup

`project_solution/` goes on `sys.path` relative to this notebook's own location.
Never hard-code an absolute path here — it works on exactly one machine, and
`tools/check_links.py` fails the build on them.

In [ ]:
import sys
import time
from datetime import date
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "project_solution"))

import dimensional_engine as de
import warehouse_live as wl

print(f"dimensional_engine loaded from: {Path(de.__file__).parent.name}/")
print(f"END_OF_TIME sentinel:           {de.END_OF_TIME}")

## 1. What does this module actually export?

Introspected from the module object, not typed out from memory — so this cell
stays correct when the implementation changes.

In [ ]:
exports = [n for n in dir(de) if not n.startswith("_") and n[0].isupper()]
print(f"{len(exports)} public types in dimensional_engine:")
for name in exports:
    obj = getattr(de, name)
    kind = "class" if isinstance(obj, type) else type(obj).__name__
    doc = (obj.__doc__ or "").strip().splitlines()
    print(f"  {name:<24} {kind:<10} {doc[0][:56] if doc else ''}")

print(f"\nwarehouse_live exports: {[n for n in dir(wl) if n[0].isupper() and not n.startswith('_')]}")

## 2. Baseline — an SCD2 dimension tiles the timeline

The property: consecutive versions of one natural key cover time with **no gap
and no overlap**, and exactly one version is open.

Watch what happens to the untracked `phone` change: it does *not* create a
version.

In [ ]:
dim = de.DimensionTable("dim_customer", scd_type=2, tracked_attributes={"state"})

ohio  = dim.upsert("C1", {"state": "OH", "phone": "555-0100"}, date(2024, 1, 1))
same  = dim.upsert("C1", {"state": "OH", "phone": "555-0999"}, date(2024, 2, 1))  # untracked
texas = dim.upsert("C1", {"state": "TX", "phone": "555-0999"}, date(2024, 6, 1))  # tracked

print(f"initial load  -> sk={ohio}")
print(f"phone change  -> sk={same}   (same key: untracked attributes never version)")
print(f"state change  -> sk={texas}   (new key)")
print(f"\nversions for C1: {dim.version_count('C1')}")

for v in dim.rows:
    end = "open" if v.is_current else v.valid_to.isoformat()
    print(f"  sk={v.surrogate_key}  {v.attributes['state']}  {v.valid_from} -> {end:<12} current={v.is_current}")

versions = [v for v in dim.rows if v.natural_key == "C1"]
gaps = [(a.valid_to, b.valid_from) for a, b in zip(versions, versions[1:])
        if (b.valid_from - a.valid_to).days != 1]
print(f"\ngaps or overlaps: {len(gaps)}   (must be 0)")
print(f"open versions:    {sum(v.is_current for v in versions)}   (must be 1)")

## 3. Predict before you run

A customer moves **OH → TX** effective 2024-06-01. Three sales fall entirely
*before* the move:

| date | customer | revenue |
| :--- | :--- | ---: |
| 2024-03-01 | C1 | 100 |
| 2024-03-01 | C2 | 50 |
| 2024-04-01 | C1 | 30 |

**Write down your answers before running the next cell:**

1. Load these three sales, aggregate by state. What do you get?
2. Now apply the June move and reprocess the **identical three sales** — an
   ordinary partition reprocess. Same answer, or different?
3. If you resolved the customer key with `current()` instead of an as-of
   lookup, what would question 2 return?

Question 2 is the requirement: **a historical window must not move when a
dimension changes afterwards.** Commit to an answer before running.

*(Note the scope. A sale dated 2024-07-01 legitimately lands in Texas once the
move is known — that is new information, not instability. The invariant is
about facts whose event dates the change does not cover.)*

In [ ]:
def build_and_aggregate(customers, products, product_sk, sales, label, resolve="as_of"):
    fact = de.FactTable("fact_sales", ("customer_sk", "product_sk"), ("revenue",))
    star = de.StarSchema(fact)
    star.add_dimension("customer_sk", customers)
    star.add_dimension("product_sk", products)
    for when, who, revenue in sales:
        if resolve == "as_of":
            sk = customers.lookup_as_of(who, when)        # correct
        else:
            sk = customers.current(who).surrogate_key     # the bug
        fact.insert({"customer_sk": sk, "product_sk": product_sk}, {"revenue": revenue}, when)
    result = star.aggregate([("customer_sk", "state")], "revenue")
    print(f"  {label:<44} { {s[0]: round(t, 2) for s, t in sorted(result.items())} }")
    return result

# Three sales, all dated before the June move.
HISTORICAL = [
    (date(2024, 3, 1), "C1", 100.0),
    (date(2024, 3, 1), "C2",  50.0),
    (date(2024, 4, 1), "C1",  30.0),
]
# Plus one after it, used later for reconciliation.
SALES = HISTORICAL + [(date(2024, 7, 1), "C1", 25.0)]

alloc = de.SurrogateKeyAllocator()
customers = de.DimensionTable("dim_customer", 2, {"state"}, alloc)
products  = de.DimensionTable("dim_product", 2, {"category"}, alloc)
product   = products.upsert("P1", {"category": "widget"}, date(2024, 1, 1))
customers.upsert("C1", {"state": "OH"}, date(2024, 1, 1))
customers.upsert("C2", {"state": "OH"}, date(2024, 1, 1))

before = build_and_aggregate(customers, products, product, HISTORICAL,
                             "first load (C1 still in OH)")

customers.upsert("C1", {"state": "TX"}, date(2024, 6, 1))
print("\n  ... C1 relocates to TX effective 2024-06-01 ...\n")

after_hist = build_and_aggregate(customers, products, product, HISTORICAL,
                                 "reprocess with an as-of join")

print(f"\n  historical window unchanged: {before == after_hist}")
print("\n  That is the requirement: re-running a partition must reproduce the")
print("  same numbers, even after the dimension has moved on.")

### Now see the wrong version fail

Same data, but resolving `current()` instead of the as-of version — the single
most common dimensional-modelling defect. Note that the *first* load is still
correct. That is why this reaches production.

In [ ]:
naive = build_and_aggregate(customers, products, product, HISTORICAL,
                            "reprocess with current() instead", resolve="current")

correct = {s[0]: round(t, 2) for s, t in sorted(after_hist.items())}
broken  = {s[0]: round(t, 2) for s, t in sorted(naive.items())}
moved   = correct.get("OH", 0) - broken.get("OH", 0)

print(f"\n  correct (as-of):   {correct}")
print(f"  broken  (current): {broken}")
print(f"\n  revenue that teleported out of Ohio: ${moved:,.2f}")
print("\n  Every dollar of it was earned in March and April, while C1 was")
print("  demonstrably in Ohio. Nothing raised. Nothing warned. And the FIRST")
print("  load of this partition was correct - only the reprocess is wrong,")
print("  which is exactly why this defect reaches production.")

## 4. Measurement — is an incremental refresh actually cheaper?

The claim in the README is that incremental refresh trades correctness
constraints for speed. Claims like that should be measured, not asserted.

In [ ]:
alloc2 = de.SurrogateKeyAllocator()
cust2 = de.DimensionTable("dim_customer", 2, {"state"}, alloc2)
prod2 = de.DimensionTable("dim_product", 2, {"category"}, alloc2)
p2 = prod2.upsert("P1", {"category": "widget"}, date(2024, 1, 1))

fact2 = de.FactTable("fact_sales", ("customer_sk", "product_sk"), ("revenue",))
star2 = de.StarSchema(fact2)
star2.add_dimension("customer_sk", cust2)
star2.add_dimension("product_sk", prod2)

STATES = ["OH", "TX", "CA", "NY"]
for i in range(400):
    sk = cust2.upsert(f"C{i}", {"state": STATES[i % 4]}, date(2024, 1, 1))
    for day in range(1, 21):
        fact2.insert({"customer_sk": sk, "product_sk": p2}, {"revenue": 10.0}, date(2024, 1, day))

def revenue_by_state(s):
    return s.aggregate([("customer_sk", "state")], "revenue")

view = de.MaterializedView("mv_revenue", star2, revenue_by_state)
view.refresh()
print(f"fact rows: {len(fact2.rows):,}    stale: {view.is_stale}")

watermark = date(2024, 1, 20)
for i in range(400):
    sk = cust2.lookup_as_of(f"C{i}", date(2024, 1, 21))
    fact2.insert({"customer_sk": sk, "product_sk": p2}, {"revenue": 5.0}, date(2024, 1, 21))
print(f"after one new day: {len(fact2.rows):,}    stale: {view.is_stale}")

t0 = time.perf_counter()
inc = view.refresh_incremental(watermark)
inc_ms = (time.perf_counter() - t0) * 1000

full_view = de.MaterializedView("mv_full", star2, revenue_by_state)
t0 = time.perf_counter()
full = full_view.refresh()
full_ms = (time.perf_counter() - t0) * 1000

print(f"\n  incremental: {inc_ms:7.3f} ms  (400 new rows)")
print(f"  full:        {full_ms:7.3f} ms  ({len(fact2.rows):,} rows)")
print(f"  speedup:     {full_ms / inc_ms:7.2f}x")
print(f"  agree:       {inc == full}   <-- cheap is worthless without this")

guarded = de.MaterializedView("mv_median", star2, revenue_by_state, supports_incremental=False)
guarded.refresh()
try:
    guarded.refresh_incremental(watermark)
except de.DimensionalModelError as exc:
    print(f"\n  non-additive view refuses: {str(exc)[:66]}...")

## 5. Reconciliation against real DuckDB

The most important cell in the notebook. The same three sales and the same SCD2
move, run once through the Python model and once through real SQL in an
embedded DuckDB. If these disagree, the model taught in Track A is lying about
how a warehouse behaves.

In [ ]:
wh = wl.LiveWarehouse(":memory:")
wh.create_schema()

# Mirror the model exactly: load both customers, apply the SCD2 move, then load
# the SAME four sales from SALES using an as-of resolution. Comparing a
# different set of sales would make the reconciliation meaningless.
wh.insert_customer_version("C1", "OH", date(2024, 1, 1))
wh.insert_customer_version("C2", "OH", date(2024, 1, 1))
live_p = wh.insert_product("P1", "widget", date(2024, 1, 1))
wh.load_scd2_change("C1", "TX", date(2024, 6, 1))

for when, who, revenue in SALES:
    wh.insert_sale(wh.resolve_customer_sk(who, when), live_p, when, revenue)

print("dim_customer in DuckDB after the two-statement SCD2 merge:")
for sk, cid, state, vf, vt, cur in wh.query(
    "SELECT customer_sk, customer_id, state, valid_from, valid_to, is_current "
    "FROM dim_customer ORDER BY customer_sk"):
    print(f"  sk={sk}  {cid}  {state}  {vf} -> {'open' if cur else str(vt):<12} current={cur}")

gaps = wh.query("""
    SELECT COUNT(*) FROM (
        SELECT valid_to, LEAD(valid_from) OVER (
            PARTITION BY customer_id ORDER BY valid_from) AS next_from
        FROM dim_customer) t
    WHERE next_from IS NOT NULL AND next_from <> valid_to + INTERVAL 1 DAY
""")[0][0]
print(f"\ngap/overlap pairs (SQL tiling check): {gaps}   (must be 0)")

model_result = build_and_aggregate(customers, products, product, SALES,
                                   "model, all 4 sales, as-of")
live_result = wh.revenue_by_state()
print(f"  {'duckdb, all 4 sales, as-of':<44} {live_result}")
print(f"\n  RECONCILED: {model_result == live_result}")
print("\n  Two independent implementations, one number. That agreement is the")
print("  only real evidence either of them is right.")

## 6. Fix this cell

The cell below asserts values that are **wrong on purpose**. Run it, read the
failure, work out the right numbers from the code above, and correct them in
place.

Two of the three are wrong. Do not change the code that computes — only the
expected values.

In [ ]:
# DELIBERATELY BROKEN - three expected values, two of them wrong. Fix in place.

expected_versions = 3          # how many versions does C1 have in `dim`?
expected_ohio     = 130.0      # OH revenue in the HISTORICAL window, as-of
expected_gaps     = 0          # gap/overlap pairs in the DuckDB dimension

actual_versions = dim.version_count("C1")
actual_ohio     = after_hist[("OH",)]
actual_gaps     = gaps

assert actual_versions == expected_versions, f"versions: expected {expected_versions}, got {actual_versions}"
assert actual_ohio     == expected_ohio,     f"OH revenue: expected {expected_ohio}, got {actual_ohio}"
assert actual_gaps     == expected_gaps,     f"gaps: expected {expected_gaps}, got {actual_gaps}"

print("All three match. Now explain WHY each number is what it is.")

## Takeaways

1. **A fact joins to the dimension version current when the event happened.**
   `current()` is not that. The bug is dormant until the first backfill, which
   is precisely why it survives review and reaches production.
2. **Closed intervals mean `valid_to = effective - 1 day.`** Off by one here
   double-counts one day per change, per key — small, compounding, and invisible
   to a spot check.
3. **Incremental refresh is valid for additive measures only.** Refuse rather
   than approximate; a plausible wrong number is worse than an exception.
4. **Idempotency has two halves.** Skip completed work, and *never* mark failed
   work complete. Implement only the first and failures become permanent.
5. **Reconcile two independent computations.** That is the only real evidence a
   number is right. "The pipeline ran successfully" is a statement about
   liveness and contains no information about correctness.

### Where to go next

- [`README.md`](README.md) — the concepts, in depth
- [`PROJECT_GUIDE.md`](PROJECT_GUIDE.md) — the 3-tier build path
- [`SELF_ASSESSMENT_AND_CHALLENGES.md`](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and 5 diagnostics
- [`TROUBLESHOOTING_AND_EDGE_CASES.md`](TROUBLESHOOTING_AND_EDGE_CASES.md) — 13 documented failure modes
- [`debug_lab/SYMPTOMS.md`](debug_lab/SYMPTOMS.md) — **6 planted defects, exit code 0.** Start here once the tests pass.
- `starter/dimensional_engine.py` — build it yourself; the 39 shipped tests are your spec